In [ ]:
import gymnasium as gym
import numpy as np
from sumo_rl import SumoEnvironment
from stable_baselines3 import DQN


class SB3CompatWrapper(gym.Wrapper):
    @staticmethod
    def _unwrap(value):
        if isinstance(value, dict):
            return next(iter(value.values()))
        return value

    @staticmethod
    def _unwrap_done(value):
        if isinstance(value, dict):
            return any(bool(v) for v in value.values())
        return bool(value)

    def reset(self, *, seed=None, options=None):
        try:
            result = self.env.reset(seed=seed)
        except TypeError:
            result = self.env.reset()

        if isinstance(result, tuple) and len(result) == 2:
            obs, info = result
            obs = self._unwrap(obs)
            if not isinstance(info, dict):
                info = {}
            return np.asarray(obs, dtype=np.float32), info

        obs = self._unwrap(result)
        return np.asarray(obs, dtype=np.float32), {}

    def step(self, action):
        result = self.env.step(action)
        if not isinstance(result, tuple):
            raise ValueError(f"Unexpected step() return format: {type(result)}")

        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            obs = np.asarray(self._unwrap(obs), dtype=np.float32)
            reward = float(self._unwrap(reward))
            terminated = self._unwrap_done(terminated)
            truncated = self._unwrap_done(truncated)
            if not isinstance(info, dict):
                info = {}
            return obs, reward, terminated, truncated, info

        if len(result) == 4:
            obs, reward, done, info = result
            obs = np.asarray(self._unwrap(obs), dtype=np.float32)
            reward = float(self._unwrap(reward))
            terminated = self._unwrap_done(done)
            truncated = False
            if not isinstance(info, dict):
                info = {}
            return obs, reward, terminated, truncated, info

        raise ValueError(f"Unexpected step() tuple length: {len(result)}")


class ExpandedSignalControlWrapper(gym.Wrapper):
    ACTION_MAP = {
        0:  ("NS", 0b001),     # NS-right
        1:  ("NS", 0b010),     # NS-straight
        2:  ("NS", 0b100),     # NS-left
        3:  ("NS", 0b011),     # NS-right+straight
        4:  ("NS", 0b101),     # NS-right+left
        5:  ("NS", 0b110),     # NS-straight+left
        6:  ("NS", 0b111),     # NS-all
        7:  ("EW", 0b001),     # EW-right
        8:  ("EW", 0b010),     # EW-straight
        9:  ("EW", 0b100),     # EW-left
        10: ("EW", 0b011),     # EW-right+straight
        11: ("EW", 0b101),     # EW-right+left
        12: ("EW", 0b110),     # EW-straight+left
        13: ("EW", 0b111),     # EW-all
        14: ("RED", 0b000),    # All-red
    }
    
    def __init__(self, env):
        super().__init__(env)
        self.action_space = gym.spaces.Discrete(15)
        self.last_action = -1
        
    def _action_to_phase(self, action):
        """Convert expanded action to base phase (0-5) for underlying env"""
        direction, movements = self.ACTION_MAP[action]
        
        if direction == "RED":
            return 0  
        if direction == "NS":
            if movements == 0b001:      
                return 0
            elif movements == 0b010:    
                return 1
            elif movements == 0b100:    
                return 2
            else: 
                if movements == 0b111:
                    return 1  
                elif movements & 0b010:
                    return 1  
                else:
                    return 0  
        else:  
            if movements == 0b001:   
                return 3
            elif movements == 0b010:    
                return 4
            elif movements == 0b100:   
                return 5
            else:  
                return 4  

    def reset(self, *, seed=None, options=None):
        self.last_action = -1
        return self.env.reset(seed=seed) if seed else self.env.reset()

    def step(self, action):
        phase = self._action_to_phase(action)        
        obs, reward, terminated, truncated, info = self.env.step(phase)
        
        self.last_action = action
        return obs, reward, terminated, truncated, info


env_base = SumoEnvironment(
    net_file="intersection.net.xml",
    route_file="intersection.rou.xml",
    out_csv_name="intersection_dqn.csv",
    use_gui=False,
    num_seconds=1000,
    single_agent=True,
)

env = ExpandedSignalControlWrapper(SB3CompatWrapper(env_base))

print(f"Training with expanded signal control action space: Discrete({env.action_space.n})")
print("\n✓ Signal Control Actions (15 total):")
print("  NS Direction (Actions 0-6):")
print("    0: Right lane only")
print("    1: Straight lane only")
print("    2: Left lane only")
print("    3: Right + Straight lanes")
print("    4: Right + Left lanes")
print("    5: Straight + Left lanes")
print("    6: ALL lanes (Right + Straight + Left)")
print("  EW Direction (Actions 7-13):")
print("    7: Right lane only")
print("    8: Straight lane only")
print("    9: Left lane only")
print("    10: Right + Straight lanes")
print("    11: Right + Left lanes")
print("    12: Straight + Left lanes")
print("    13: ALL lanes (Right + Straight + Left)")
print("  Safety (Action 14):")
print("    14: All-red (transition phase)")
print("\n✓ Traffic: 50% reduced volume with dynamic variability")
print("   - Morning (0-150s): Peak NS, lower EW")
print("   - Afternoon (150-300s): Mixed patterns")
print("   - Evening (300-450s): Different demand ratios")

model = DQN(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=0.001,
    buffer_size=5000,
    learning_starts=100,
    batch_size=32,
    gamma=0.99,
)
print("\n🚦 Starting DQN training with expanded signal flexibility...")
model.learn(total_timesteps=20000)
model.save("dqn_intersection")
env.close()
print("✓ Training completed!")

Training with expanded signal control action space: Discrete(15)

✓ Signal Control Actions (15 total):
  NS Direction (Actions 0-6):
    0: Right lane only
    1: Straight lane only
    2: Left lane only
    3: Right + Straight lanes
    4: Right + Left lanes
    5: Straight + Left lanes
    6: ALL lanes (Right + Straight + Left)
  EW Direction (Actions 7-13):
    7: Right lane only
    8: Straight lane only
    9: Left lane only
    10: Right + Straight lanes
    11: Right + Left lanes
    12: Straight + Left lanes
    13: ALL lanes (Right + Straight + Left)
  Safety (Action 14):
    14: All-red (transition phase)

✓ Traffic: 50% reduced volume with dynamic variability
   - Morning (0-150s): Peak NS, lower EW
   - Afternoon (150-300s): Mixed patterns
   - Evening (300-450s): Different demand ratios
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

🚦 Starting DQN training with expanded signal flexibility...
----------------------------------

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import DQN
from sumo_rl import SumoEnvironment


class SB3CompatWrapper(gym.Wrapper):
    @staticmethod
    def _unwrap(value):
        if isinstance(value, dict):
            return next(iter(value.values()))
        return value

    @staticmethod
    def _unwrap_done(value):
        if isinstance(value, dict):
            return any(bool(v) for v in value.values())
        return bool(value)

    def reset(self, *, seed=None, options=None):
        try:
            result = self.env.reset(seed=seed)
        except TypeError:
            result = self.env.reset()

        if isinstance(result, tuple) and len(result) == 2:
            obs, info = result
            obs = self._unwrap(obs)
            if not isinstance(info, dict):
                info = {}
            return np.asarray(obs, dtype=np.float32), info

        obs = self._unwrap(result)
        return np.asarray(obs, dtype=np.float32), {}

    def step(self, action):
        result = self.env.step(action)
        if not isinstance(result, tuple):
            raise ValueError(f"Unexpected step() return format: {type(result)}")

        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            obs = np.asarray(self._unwrap(obs), dtype=np.float32)
            reward = float(self._unwrap(reward))
            terminated = self._unwrap_done(terminated)
            truncated = self._unwrap_done(truncated)
            if not isinstance(info, dict):
                info = {}
            return obs, reward, terminated, truncated, info

        if len(result) == 4:
            obs, reward, done, info = result
            obs = np.asarray(self._unwrap(obs), dtype=np.float32)
            reward = float(self._unwrap(reward))
            terminated = self._unwrap_done(done)
            truncated = False
            if not isinstance(info, dict):
                info = {}
            return obs, reward, terminated, truncated, info

        raise ValueError(f"Unexpected step() tuple length: {len(result)}")


class ExpandedSignalControlWrapper(gym.Wrapper):
    """Maps 15 expanded signal actions to 6 base phases."""

    ACTION_MAP = {
        0: ("NS", 0b001),
        1: ("NS", 0b010),
        2: ("NS", 0b100),
        3: ("NS", 0b011),
        4: ("NS", 0b101),
        5: ("NS", 0b110),
        6: ("NS", 0b111),
        7: ("EW", 0b001),
        8: ("EW", 0b010),
        9: ("EW", 0b100),
        10: ("EW", 0b011),
        11: ("EW", 0b101),
        12: ("EW", 0b110),
        13: ("EW", 0b111),
        14: ("RED", 0b000),
    }

    def __init__(self, env):
        super().__init__(env)
        self.action_space = gym.spaces.Discrete(15)

    @staticmethod
    def _to_int_action(action):
        if isinstance(action, np.ndarray):
            return int(action.item())
        return int(action)

    def _action_to_phase(self, action):
        action = self._to_int_action(action)
        direction, movements = self.ACTION_MAP[action]
        if direction == "RED":
            return 0
        if direction == "NS":
            if movements == 0b001:
                return 0
            if movements == 0b010:
                return 1
            if movements == 0b100:
                return 2
            return 1
        if movements == 0b001:
            return 3
        if movements == 0b010:
            return 4
        if movements == 0b100:
            return 5
        return 4

    def reset(self, *, seed=None, options=None):
        return self.env.reset(seed=seed) if seed is not None else self.env.reset()

    def step(self, action):
        phase = self._action_to_phase(action)
        return self.env.step(phase)


# Create evaluation environment
env_base = SumoEnvironment(
    net_file="intersection.net.xml",
    route_file="intersection.rou.xml",
    out_csv_name="intersection_dqn_eval.csv",
    use_gui=False,
    num_seconds=1000,
    single_agent=True,
)

env = ExpandedSignalControlWrapper(SB3CompatWrapper(env_base))
model = DQN.load("dqn_intersection")

obs, info = env.reset()
counter = 0
terminated = False
truncated = False
total_reward = 0.0

print("Evaluating trained DQN on dynamic traffic with expanded signal control...")
while not (terminated or truncated) and counter < 10000:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += float(reward)
    counter += 1

print("Evaluation completed!")
print(f"Steps: {counter}")
print(f"Total reward: {total_reward:.2f}")
env.close()

Evaluating trained DQN on dynamic traffic with expanded signal control...
Evaluation completed!
Steps: 200
Total reward: -20.50


In [22]:
import gymnasium as gym
import numpy as np
from pathlib import Path
from stable_baselines3 import DQN
from sumo_rl import SumoEnvironment


class SB3CompatWrapper(gym.Wrapper):
    @staticmethod
    def _unwrap(value):
        if isinstance(value, dict):
            return next(iter(value.values()))
        return value

    @staticmethod
    def _unwrap_done(value):
        if isinstance(value, dict):
            return any(bool(v) for v in value.values())
        return bool(value)

    def reset(self, *, seed=None, options=None):
        try:
            result = self.env.reset(seed=seed)
        except TypeError:
            result = self.env.reset()

        if isinstance(result, tuple) and len(result) == 2:
            obs, info = result
            return np.asarray(self._unwrap(obs), dtype=np.float32), (info if isinstance(info, dict) else {})

        return np.asarray(self._unwrap(result), dtype=np.float32), {}

    def step(self, action):
        result = self.env.step(action)
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
        else:
            obs, reward, done, info = result
            terminated, truncated = done, False

        obs = np.asarray(self._unwrap(obs), dtype=np.float32)
        reward = float(self._unwrap(reward))
        terminated = self._unwrap_done(terminated)
        truncated = self._unwrap_done(truncated)
        if not isinstance(info, dict):
            info = {}
        return obs, reward, terminated, truncated, info


class ExpandedSignalControlWrapper(gym.Wrapper):
    ACTION_MAP = {
        0: ("NS", 0b001),
        1: ("NS", 0b010),
        2: ("NS", 0b100),
        3: ("NS", 0b011),
        4: ("NS", 0b101),
        5: ("NS", 0b110),
        6: ("NS", 0b111),
        7: ("EW", 0b001),
        8: ("EW", 0b010),
        9: ("EW", 0b100),
        10: ("EW", 0b011),
        11: ("EW", 0b101),
        12: ("EW", 0b110),
        13: ("EW", 0b111),
        14: ("RED", 0b000),
    }

    def __init__(self, env):
        super().__init__(env)
        self.action_space = gym.spaces.Discrete(15)

    @staticmethod
    def _to_int_action(action):
        if isinstance(action, np.ndarray):
            return int(action.item())
        return int(action)

    def _action_to_phase(self, action):
        action = self._to_int_action(action)
        direction, movements = self.ACTION_MAP[action]
        if direction == "RED":
            return 0
        if direction == "NS":
            if movements == 0b001:
                return 0
            if movements == 0b010:
                return 1
            if movements == 0b100:
                return 2
            return 1
        if movements == 0b001:
            return 3
        if movements == 0b010:
            return 4
        if movements == 0b100:
            return 5
        return 4

    def reset(self, *, seed=None, options=None):
        return self.env.reset(seed=seed) if seed is not None else self.env.reset()

    def step(self, action):
        phase = self._action_to_phase(action)
        return self.env.step(phase)


def find_project_dir() -> Path:
    cwd = Path.cwd()
    direct = cwd / "intersection.net.xml"
    if direct.exists():
        return cwd
    rl_dir = cwd / "RL"
    if (rl_dir / "intersection.net.xml").exists():
        return rl_dir
    matches = list(cwd.rglob("intersection.net.xml"))
    if matches:
        return matches[0].parent
    raise FileNotFoundError("Could not find intersection.net.xml from current working directory")


project_dir = find_project_dir()
net_file = str((project_dir / "intersection.net.xml").resolve())
route_file = str((project_dir / "intersection.rou.xml").resolve())
out_csv = str((project_dir / "intersection_dqn_visual_eval.csv").resolve())
model_path = str((project_dir / "dqn_intersection.zip").resolve())

print("Using project directory:", project_dir)
print("Net file:", net_file)
print("Route file:", route_file)
print("Model file:", model_path)

vis_env_base = SumoEnvironment(
    net_file=net_file,
    route_file=route_file,
    out_csv_name=out_csv,
    use_gui=True,
    num_seconds=1000,
    single_agent=True,
    sumo_warnings=False,
    additional_sumo_cmd="--delay 80",
)

vis_env = ExpandedSignalControlWrapper(SB3CompatWrapper(vis_env_base))
vis_model = DQN.load(model_path)
obs, info = vis_env.reset()
terminated = False
truncated = False
steps = 0

cum_reward = 0.0

# Metrics tracking
import traci
wait_times = []
max_wait_time = 0.0

print("\nRunning visual evaluation with dynamic traffic...")
print("(SUMO GUI window will display signal control decisions)")
while not (terminated or truncated) and steps < 5000:
    action, _ = vis_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = vis_env.step(action)
    cum_reward += reward
    steps += 1
    
    # Collect wait time metrics from all vehicles
    try:
        vehicle_ids = traci.vehicle.getIDList()
        for veh_id in vehicle_ids:
            wait_time = traci.vehicle.getWaitingTime(veh_id)
            if wait_time > 0:
                wait_times.append(wait_time)
                max_wait_time = max(max_wait_time, wait_time)
    except:
        pass  # TraCI might not be available in all execution contexts

print("\n✓ Visual evaluation finished!")
print(f"  Simulation Steps: {steps}")
print(f"  Cumulative Reward: {round(cum_reward, 3)}")
print(f"  Average Reward per Step: {round(cum_reward / max(steps, 1), 4)}")

# Report wait time statistics
if wait_times:
    avg_wait_time = np.mean(wait_times)
    print(f"\n✓ Traffic Performance Metrics:")
    print(f"  Average Wait Time: {round(avg_wait_time, 2)} seconds")
    print(f"  Max Wait Time: {round(max_wait_time, 2)} seconds")
    print(f"  Total Vehicles with Wait: {len(wait_times)}")
else:
    print("\n✓ Traffic Performance Metrics:")
    print(f"  Average Wait Time: 0.00 seconds (all vehicles flowing)")
    print(f"  Max Wait Time: 0.00 seconds")

vis_env.close()

Using project directory: c:\Users\Dhruv\Desktop\workshop\study\miscellenous\llms\RL
Net file: C:\Users\Dhruv\Desktop\workshop\study\miscellenous\llms\RL\intersection.net.xml
Route file: C:\Users\Dhruv\Desktop\workshop\study\miscellenous\llms\RL\intersection.rou.xml
Model file: C:\Users\Dhruv\Desktop\workshop\study\miscellenous\llms\RL\dqn_intersection.zip

Running visual evaluation with dynamic traffic...
(SUMO GUI window will display signal control decisions)

✓ Visual evaluation finished!
  Simulation Steps: 200
  Cumulative Reward: -22.31
  Average Reward per Step: -0.1115

✓ Traffic Performance Metrics:
  Average Wait Time: 37.04 seconds
  Max Wait Time: 273.0 seconds
  Total Vehicles with Wait: 11173
